In [1]:
from data.cleaning import read_csv
from core import Config
import pandas as pd

config = Config()
all_df: pd.DataFrame = read_csv(
    filepath=config.data_dir / "datasets" / "median" / "median_imputed.csv",
    dtypes_filepath=config.data_dir / "datasets" / "median" / "median_imputed_dtypes.csv",
    index_list=[0]
)

In [ ]:
print("Original DataFrame:\n", all_df)
print("\nDataFrame Info:\n")
all_df.info()
print("\nDescriptive Statistics:\n", all_df.describe())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.boxplot(x=all_df['TR.AnalyticAntiTakeoverDevicesScore'])
plt.title('Box Plot of Values')
plt.xlabel('Values')
plt.show()

In [ ]:

from scipy.stats import zscore

# Calculate Z-scores for the 'values' column
all_df['z_score'] = np.abs(zscore(all_df['TR.AnalyticAntiTakeoverDevicesScore']))

# Define a threshold (common thresholds are 2, 2.5, or 3)
threshold_z = 1

# Identify outliers based on the Z-score
outliers_zscore = all_df[all_df['z_score'] > threshold_z]

print("Outliers using Z-score method (threshold=", threshold_z, "):\n", outliers_zscore)


In [2]:
from typing import Callable

def flag_outliers_per_instrument(df: pd.DataFrame, value_cols: list[str], instrument_col="Instrument", factor=1.5):
    df = df.copy()
    mask = pd.Series(True, index=df.index)

    for col in value_cols:
        grouped = df.groupby(instrument_col)[col]
        quantile_25: Callable[[pd.Series], float] = lambda s: s.quantile(0.25)
        quantile_75: Callable[[pd.Series], float] = lambda s: s.quantile(0.75)
        q1 = grouped.transform(quantile_25)
        q3 = grouped.transform(quantile_75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        mask &= (df[col] >= lower) & (df[col] <= upper)

    return mask  # True = keep, False = outlier

# usage:
num_cols = all_df.select_dtypes(include=['number']).columns.tolist()
keep_mask: pd.Series = flag_outliers_per_instrument(all_df, num_cols, "Instrument", factor=1.5)
df_clean = all_df[keep_mask]